# RS04 · Surviving power cuts and hangs - commit logs, watchdogs, reset causes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS04_surviving_power_cuts_and_hangs_commit_logs_watch.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 60 min

**Skill Hub concepts:** Append-only log with commit markers; Watchdog supervision; Reset cause and degraded mode; Post-mortem debugging

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS04")

## Map: what happens at the worst possible moment

**Append-only log with a commit marker** (flash erased to `0xFF`):
```
 [0xA5 magic][len][payload ... len bytes][crc8 over len+payload][commit]     commit: 0xFF = torn, 0x00 = valid
 write order: magic, len, payload, crc  ->  then, in a separate write, commit = 0x00
```
A power cut before the commit byte leaves a record the scanner skips; nothing half-written is ever trusted.

**Watchdog supervision:** a hardware watchdog (IWDG) resets the chip unless kicked in time. Kicking from a timer
interrupt proves nothing. A supervisor task kicks **only if every critical task checked in recently**, so a stuck task
causes a reset.

| STM32G4 `RCC_CSR` flag | Bit | Meaning |
|---|---|---|
| LPWRRSTF | 31 | low-power reset |
| WWDGRSTF | 30 | window watchdog |
| IWDGRSTF | 29 | independent watchdog |
| SFTRSTF | 28 | software reset (`NVIC_SystemReset`) |
| BORRSTF | 27 | brown-out |
| PINRSTF | 26 | reset pin |
| OBLRSTF | 25 | option byte load |

Read the flags at boot (then clear them with RMVF), log them, and pick a mode: repeated watchdog resets mean the
firmware is crashing in a loop, so boot **degraded** or **safe** (motors disabled) instead of crashing forever.
**Post-mortem debugging:** the fault handler (lab C06) saves PC, LR, CFSR and a timestamp into a no-init RAM section
or this log before resetting, so you can read it after the fact.

IWDG timeout: $t = \dfrac{\text{prescaler} \times (\text{reload} + 1)}{f_{LSI}}$, often approximated as $\text{prescaler} \times \text{reload} / f_{LSI}$.

In [ ]:
import random

def crc8(data):
    c = 0
    for b in data:
        c ^= b
        for _ in range(8):
            c = ((c << 1) ^ 0x07) & 0xFF if c & 0x80 else (c << 1) & 0xFF
    return c

class PowerCut(Exception):
    pass

class LogFlash:
    """4 KB of erased flash; program() can only clear bits, and may lose power after `budget` bytes."""
    def __init__(self, size=4096, budget=None):
        self.mem, self.budget = bytearray(b"\xff" * size), budget
    def program(self, addr, data):
        for i, b in enumerate(data):
            if self.budget is not None:
                if self.budget == 0:
                    raise PowerCut()
                self.budget -= 1
            self.mem[addr + i] &= b

## Task 1: append and recover

In [ ]:
def append(flash, offset, payload):
    """Write one record at `offset`; return the offset after it."""
    raise NotImplementedError  # TODO

def recover(flash):
    """Return (list of committed payloads, next free offset)."""
    raise NotImplementedError  # TODO

lab.check("log", append, recover)

In [ ]:
%%writefile rs04_log.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/rs04.h
// Grade it with lab.check("log", "rs04_log.c"). For C++, name the file .cpp in both places.
#include "rs04.h"

int32_t log_append(const LogFlashOps *flash, uint32_t offset, const uint8_t *payload, size_t len) {
    (void)flash; (void)offset; (void)payload; (void)len;
    return -1; /* TODO: magic, length, payload, crc8, then the commit byte in a separate program call */
}

size_t log_recover(const uint8_t *mem, size_t size, uint8_t *out, size_t out_cap, size_t *lens, size_t max_records, uint32_t *next_offset) {
    (void)mem; (void)size; (void)out; (void)out_cap; (void)lens; (void)max_records;
    *next_offset = 0;
    return 0; /* TODO: walk records; keep the committed ones; skip torn ones; report where to append next */
}

In [ ]:
# lab.check("log", "rs04_log.c")  # uncomment after writing the C/C++ cell above

## Task 2: the watchdog supervisor

In [ ]:
class Supervisor:
    def __init__(self, max_interval):        # {task_name: seconds}
        self.max_interval = dict(max_interval)
        self.last = {t: None for t in max_interval}

    def checkin(self, task, now):
        self.last[task] = now

    def kick(self, now):
        """True if the hardware watchdog may be refreshed now."""
        raise NotImplementedError  # TODO

lab.check("supervisor", Supervisor)

In [ ]:
%%writefile rs04_supervisor.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/rs04.h
// Grade it with lab.check("supervisor", "rs04_supervisor.c"). For C++, name the file .cpp in both places.
#include "rs04.h"

void sup_init(Supervisor *s, const double *max_interval, size_t n) {
    s->n = n < SUP_MAX_TASKS ? n : SUP_MAX_TASKS;
    for (size_t i = 0; i < s->n; ++i) { s->max_interval[i] = max_interval[i]; s->last[i] = 0; s->seen[i] = false; }
}

void sup_checkin(Supervisor *s, size_t task, double now) { (void)s; (void)task; (void)now; /* TODO */ }

bool sup_kick(const Supervisor *s, double now) { (void)s; (void)now; return false; /* TODO */ }

In [ ]:
# lab.check("supervisor", "rs04_supervisor.c")  # uncomment after writing the C/C++ cell above

## Task 3: reset causes and the boot mode
`history` is a list of cause sets, oldest first, newest last. Three watchdog resets (IWDG) in a row -> `"safe"`;
the newest one caused by IWDG or WWDG -> `"degraded"`; otherwise `"normal"`.

In [ ]:
FLAGS = {31: "LPWR", 30: "WWDG", 29: "IWDG", 28: "SFT", 27: "BOR", 26: "PIN", 25: "OBL"}

def decode_csr(csr):
    raise NotImplementedError  # TODO

def boot_mode(history):
    raise NotImplementedError  # TODO

lab.check("resets", decode_csr, boot_mode)

In [ ]:
%%writefile rs04_resets.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/rs04.h
// Grade it with lab.check("resets", "rs04_resets.c"). For C++, name the file .cpp in both places.
#include "rs04.h"

size_t decode_csr(uint32_t csr, const char **names, size_t cap) {
    (void)csr; (void)names; (void)cap;
    return 0; /* TODO */
}

int boot_mode(const uint32_t *history_csr, size_t n) {
    (void)history_csr; (void)n;
    return BOOT_NORMAL; /* TODO: three consecutive IWDG resets -> safe; last reset by a watchdog -> degraded */
}

In [ ]:
# lab.check("resets", "rs04_resets.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** IWDG on a 32 kHz LSI, prescaler 64, reload 1000 (use prescaler x reload / f). Timeout in seconds?

**M2 (Apply).** How many 32-byte records fit in one 4096-byte sector?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Evaluate).** A teammate kicks the IWDG from the 1 kHz SysTick interrupt. Problem?
- a) it wears out the watchdog
- b) the interrupt keeps firing even when the main loop or a task is stuck, so the watchdog never catches the hang
- c) nothing

**K2 (Analyze).** A robot in the field reboots every few hours. List what your firmware should have recorded so you can
diagnose it without reproducing it, and where you keep it across the reset.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs04.h`).
3. In `roboprep/checks/rs04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()